# Chromasense Colab Runner

Runner notebook only. Core logic stays in repo files.

Use Colab GPU runtime, preferably T4: `Runtime > Change runtime type > T4 GPU`.

## 1. Configure Repo

Set `REPO_URL` if project is on GitHub. Leave blank to upload a zip of the repo folder.

In [ ]:
REPO_URL = ""  # Example: "https://github.com/your-name/chromasense.git"
BRANCH = "main"
PROJECT_DIR = "/content/chromasense"
PORT = 8000
NGROK_DOMAIN = None  # Optional reserved domain, e.g. "your-domain.ngrok-free.app"

## 2. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print("CUDA:", torch.cuda.get_device_name(0))
else:
    print("CUDA not available. Switch Colab runtime to GPU for Qwen.")

## 3. Clone Or Upload Repo

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import zipfile

project_dir = Path(PROJECT_DIR)
protected_dirs = {Path("/"), Path("/content"), Path.home()}
if project_dir.resolve() in {path.resolve() for path in protected_dirs}:
    raise ValueError(f"Refusing to replace protected path: {project_dir}")

if project_dir.exists():
    shutil.rmtree(project_dir)

if REPO_URL.strip():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, PROJECT_DIR],
        check=True,
    )
else:
    from google.colab import files

    print("Upload a zip of the repo folder.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise RuntimeError("No zip uploaded.")

    extract_root = Path("/content/chromasense_upload")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_names[0]) as archive:
        archive.extractall(extract_root)

    candidates = [path for path in extract_root.rglob("main.py") if (path.parent / "requirements.txt").exists()]
    if not candidates:
        raise RuntimeError("Uploaded zip must contain main.py and requirements.txt.")
    shutil.copytree(candidates[0].parent, project_dir)

os.chdir(project_dir)
print("Project ready:", project_dir)

## 4. Install Dependencies

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

## 5. Configure Ngrok

Recommended: add Colab secret named `NGROK_AUTHTOKEN`. Public demo can run without app auth.

In [ ]:
from pyngrok import ngrok

try:
    from google.colab import userdata
    token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    token = None

if token:
    ngrok.set_auth_token(token)
    print("ngrok token configured.")
else:
    print("No NGROK_AUTHTOKEN secret found. If tunnel fails, add token in Colab secrets and rerun this cell.")

## 6. Start FastAPI

In [ ]:
import subprocess
import sys
import time
import requests

server_log_path = Path("/content/chromasense_server.log")
server_log = server_log_path.open("w", encoding="utf-8")

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=PROJECT_DIR,
    stdout=server_log,
    stderr=subprocess.STDOUT,
    text=True,
)

base_url = f"http://127.0.0.1:{PORT}"
for _ in range(60):
    try:
        response = requests.get(base_url + "/", timeout=2)
        if response.ok:
            print("Server ready:", response.json())
            break
    except Exception:
        time.sleep(2)
else:
    server.terminate()
    server_log.close()
    raise RuntimeError("FastAPI server did not start. Check server output cell below.")

## 7. Open Public Tunnel

In [ ]:
ngrok.kill()

connect_kwargs = {"addr": PORT, "proto": "http"}
if NGROK_DOMAIN:
    connect_kwargs["domain"] = NGROK_DOMAIN

public_url = ngrok.connect(**connect_kwargs).public_url
print("API:", public_url)
print("Frontend:", public_url + "/app")
print("Docs:", public_url + "/docs")

## 8. Warm Up Models

This can take several minutes on first run while models download/load.

In [ ]:
import requests

warmup_response = requests.post(public_url + "/warmup", timeout=1800)
print("Status:", warmup_response.status_code)
print(warmup_response.json())

## 9. Demo

Open printed `/app` URL. Upload image, change `n_colors`, run analysis, show palette, mood, guidance, warnings, metadata.

In [ ]:
from IPython.display import HTML, display

display(HTML(f'<p><a href="{public_url}/app" target="_blank">Open Chromasense app</a></p>'))

## 10. Optional: View Server Logs

In [ ]:
server_log.flush()
print(Path("/content/chromasense_server.log").read_text(encoding="utf-8")[-12000:])

## 11. Stop Server

In [ ]:
server.terminate()
server_log.close()
ngrok.kill()
print("Stopped.")